In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('../pca_full')  # Adjust the path as needed
from pca_full import pca_full

# Ensure reproducibility
np.random.seed(42)

# Set default text interpreter to LaTeX (for matplotlib)
plt.rcParams['text.usetex'] = True

# Define the OHspecial_transform function
def OHspecial_transform(df):
    names = df.columns.values
    data = df.values
    labels = []
    
    for j in range(len(names)):
        col_data = data[:, j]
        temp = np.unique(col_data[~np.isnan(col_data)])
        temp = temp[temp >= 0]
        
        if len(temp) == 0:
            # All data in column j is NaN or negative
            var = col_data.reshape(-1, 1)
            label = [names[j]]
        elif len(temp) <= 2:
            # Keep original data
            var = col_data.reshape(-1, 1)
            label = [names[j]]
        else:
            # One-hot encode
            var = np.zeros((data.shape[0], len(temp)))
            for l in range(len(temp)):
                var[:, l] = np.where(col_data == temp[l], 1, 0)
            idx = np.where(np.isnan(col_data))[0]
            var[idx, :] = np.nan
            # Use str instead of int to match MATLAB's num2str behavior
            label = [f"{names[j]}_{str(temp[l])}" for l in range(len(temp))]
        
        if j == 0:
            OH = var
        else:
            OH = np.hstack((OH, var))
        labels.extend(label)
    
    OH_df = pd.DataFrame(OH, columns=labels)
    return OH_df

# Read in data and remove non-cultural features + features missing > 50% of possible values

# From EA
KinOrgRaw = pd.read_csv('KinshipOrgDataRaw.csv')
SubRaw = pd.read_csv('SubsistenceDataRaw.csv')
# EA_All_Raw = pd.read_csv('EA_All.csv')  # Commented out in MATLAB code

# From Pulotu
IsoRaw = pd.read_csv('IsoDataRaw.csv')
ReligRaw = pd.read_csv('RelDataRaw.csv')

# Other datasets
Binford = pd.read_csv('Binford_all.csv')
Seshat = pd.read_csv('Seshat.csv')
Birds = pd.read_csv('data_birds_OH.csv')

# Apply OHspecial_transform to datasets
KinOrg = OHspecial_transform(KinOrgRaw)
Sub = OHspecial_transform(SubRaw)
Iso = OHspecial_transform(IsoRaw)
Relig = OHspecial_transform(ReligRaw)

for w in range(2, 3):

    if w == 1:
        X = KinOrg
    elif w == 2:
        X = Sub
    elif w == 3:
        X = Relig
    elif w == 4:
        X = Iso
    elif w == 5:
        X = Binford
    elif w == 6:
        X = Seshat
    elif w == 7:
        X = Birds

    # Remove features missing > 50% of their entries
    dim = X.shape
    X = X.iloc[:, 0:dim[1]]  # Redundant but kept for similarity
    cols = X.columns.values
    X = X.to_numpy()

    counts = []
    for j in range(dim[1]):
        temp = np.isnan(X[:, j]).astype(int)
        idx = np.where(temp == 1)[0]
        counts.append(len(idx))

    counts = np.array(counts) / dim[0]
    idx = np.where(counts > 0.5)[0]
    cols = np.delete(cols, idx)
    X = np.delete(X, idx, axis=1)
    dim = X.shape

    # Compute initial mean and standard deviation, ignoring NaNs
    MnInit = np.tile(np.nanmean(X, axis=0), (dim[0], 1))
    STDInit = np.tile(np.nanstd(X, axis=0), (dim[0], 1))

    if w < 5:
        X_ = X - MnInit
    elif w == 5:
        X_ = X
    elif w == 6:
        X_ = (X - MnInit) / STDInit
    elif w == 7:
        X_ = X

    X_ = X_.T

    MaxStop = min(dim)
    if MaxStop > 100:
        MaxStop = 100
    zz = np.arange(2, MaxStop + 1)

    # Initialize lists for dynamic data collection
    rms = []
    accu = []
    rmsA = []
    costA = []
    var_exp = []

    for y in range(MaxStop - 1):
        if y == 34:
            z = zz[y]
            print('###############################')
            print(f'Number of PCs: {z}')
            print('###############################')
            if y > 0:
                time.sleep(1)
    
            for k in range(1):  # Change if you want to do replications
    
                opts = {
                    'maxiters': 80,
                    'algorithm': 'vb',
                    'uniquesv': 0,
                    'rmsstop': [80, np.finfo(float).eps, np.finfo(float).eps],
                    'cfstop': [80, 0, 0],
                    'minangle': 0
                }
    
                result = pca_full(np.copy(X_), z, **opts)
                A = result['A']
                S = result['S']
                Mu = result['Mu']
                V = result['V']
                cv = result['cv']
                hp = result['hp']
                lc = result['lc']
    
                Xrec = Mu + A @ S
    
                # Compute accuracy
                accu_num = np.abs(np.round(Xrec.T + MnInit) - X)
                accu_num = len(np.where(accu_num > 0)[0])
                accu_value = 1 - accu_num / len(np.where(~np.isnan(X_.flatten()))[0])
                accu.append(accu_value)
    

    
                # Compute explained variance
                vv = np.zeros(MaxStop)
                v = np.linalg.eigvals(np.cov(Xrec))
                v = np.flipud(np.sort(np.real(v)))
                v = v[:MaxStop]
                v = np.round(v, 4)
                v = v / np.sum(v)
                vv[:len(v)] = v
                var_exp.append(vv)
    
                break

    # After the loop, convert lists to NumPy arrays
    rmsA = np.array(rmsA)
    costA = np.array(costA)
    var_exp = np.array(var_exp)
    rms = np.array(rms)
    accu = np.array(accu)
    
    AFin = np.array(result['A'])
    SFin = np.array(result['S'])
    MuFin = np.array(result['Mu']).flatten()
    VFin = result['V']
    cvFin = result['cv']
    hpFin = result['hp']
    lcFin = result['lc']
    Xrec = MuFin[:, np.newaxis] + AFin @ SFin
    
    # Compute Vr
    Vr = np.zeros_like(X_)
    for i in range(X_.shape[0]):
        for j in range(X_.shape[1]):
            Vr[i, j] = (AFin[i, :] @ cvFin['S'][j] @ AFin[i, :].T +
                        SFin[:, j].T @ cvFin['A'][i] @ SFin[:, j] +
                        np.sum(cvFin['S'][j] * cvFin['A'][i]) +
                        cvFin['Mu'][i])
    VrFin = Vr
  




###############################
Number of PCs: 36
###############################
No empty rows or columns
Step 0: rms = 0.861404
Step 1: cost = 24621.073742, rms = 0.624468, angle = 1.39e+00 (3e-01 sec)
Step 2: cost = 20974.854657, rms = 0.291311, angle = 5.53e-01 (2e-01 sec)
Step 3: cost = 19102.669844, rms = 0.279563, angle = 3.50e-01 (2e-01 sec)
Step 4: cost = 18361.415192, rms = 0.276877, angle = 3.52e-01 (4e-01 sec)
Step 5: cost = 18021.454350, rms = 0.265832, angle = 3.27e-01 (2e-01 sec)
Step 6: cost = 17787.861795, rms = 0.255411, angle = 9.11e-01 (2e-01 sec)
Step 7: cost = 17511.325573, rms = 0.238678, angle = 1.19e+00 (3e-01 sec)
Step 8: cost = 17253.037611, rms = 0.224801, angle = 8.24e-01 (3e-01 sec)
Step 9: cost = 17079.590289, rms = 0.215736, angle = 1.32e+00 (2e-01 sec)
Step 10: cost = 16935.072391, rms = 0.207426, angle = 1.41e+00 (2e-01 sec)
Step 11: cost = 16755.986511, rms = 0.196691, angle = 9.76e-01 (3e-01 sec)
Step 12: cost = 16590.586452, rms = 0.187380, angle = 

In [2]:
class TestStat:
    import torch
    import numpy as np

    def __init__(self, Xrec, S, Vr, reps, alpha):
        self.Xrec = self._to_tensor(Xrec)
        self.S = self._to_tensor(S)
        self.Vr = self._to_tensor(Vr)
        self.reps = reps
        self.alpha = alpha
        self.m, self.n = self.Xrec.shape
        self.q, self.rr = self.S.shape

    def _to_tensor(self, array):
        if isinstance(array, self.np.ndarray):
            return self.torch.tensor(array)
        return array

    def get_matrix(self):
        mu = self.Xrec
        sigma = self.torch.sqrt(self.Vr)
        r = self.torch.normal(mu, sigma)
        return r

    def test_stat(self):
        All_vals = self.torch.zeros((self.reps, self.q-1))

        for l in range(self.reps):
            r = self.get_matrix()
            # vals = self.torch.linalg.eigvals(r.T @ r)
            # vals = self.torch.sort(vals, descending=True).values[:self.q-1]
            vals = self.torch.linalg.eigvals(r.T @ r)
            vals = vals.real  # Take just the real part
            vals = self.torch.sort(vals, descending=True).values[:self.q-1]

            scale = (self.torch.arange(self.q-1, 0, -1) * vals) / self.torch.cumsum(vals.flip(0), dim=0).flip(0)

            All_vals[l, :] = scale

            if l % (self.reps // 100) == 0:
                print(f'{(l/self.reps)*100:.2f} percent complete')

        # vals = self.torch.linalg.eigvals(self.Xrec.T @ self.Xrec)
        # vals = self.torch.sort(vals, descending=True).values[:self.q-1
        vals = self.torch.linalg.eigvals(self.Xrec.T @ self.Xrec)
        vals = vals.real  # Take just the real part
        vals = self.torch.sort(vals, descending=True).values[:self.q-1]

        scale = (self.torch.arange(self.q-1, 0, -1) * vals) / self.torch.cumsum(vals.flip(0), dim=0).flip(0)

        # p = self.torch.mean(All_vals >= scale, dim=0)
        p = self.torch.mean((All_vals >= scale).float(), dim=0)

        phat_p = self.torch.zeros(self.q-1)

        for j in range(1, self.q-1):
            phat_p[j] = p[j] / (self.q - j)
            if phat_p[j] > self.alpha:
                break

        if phat_p[-1] < 1.2 * self.alpha:
            phat_o = self.torch.zeros(self.q)
            for j in range(self.q-1):
                # phat_o[j] = self.torch.mean(All_vals[:, j] >= scale[j]) / (self.q - j)
                phat_o[j] = self.torch.mean((All_vals[:, j] >= scale[j]).float()) / (self.q - j)
        else:
            phat_o = self.torch.nan

        # return [phat_p, phat_o] if len(phat_o) > 1 else phat_p
        if isinstance(phat_o, self.torch.Tensor) and phat_o.numel() > 1:
            return [phat_p, phat_o]
        else:
            return phat_p


In [5]:
import numpy as np

reps = 10000
alpha = 0.05

# Instantiate the TestStat class
test_stat_instance = TestStat(Xrec=Xrec, S=S, Vr=Vr, reps=reps, alpha=alpha)

# Call the test_stat method
results = test_stat_instance.test_stat()

# 'results' will be either a tensor representing phat_p or a list [phat_p, phat_o]
print("Results:", results)

if isinstance(results, list):
    phat_p, phat_o = results
else:
    phat_p = results

phat_p = phat_p.numpy()
alpha = 0.05
significant_indices = (phat_p <= alpha).nonzero()[0]
num_significant_pcs = len(significant_indices)

print("Significant PC indices (0-based):", significant_indices)
print("Number of significant PCs:", num_significant_pcs)


0.00 percent complete
1.00 percent complete
2.00 percent complete
3.00 percent complete
4.00 percent complete
5.00 percent complete
6.00 percent complete
7.00 percent complete
8.00 percent complete
9.00 percent complete
10.00 percent complete
11.00 percent complete
12.00 percent complete
13.00 percent complete
14.00 percent complete
15.00 percent complete
16.00 percent complete
17.00 percent complete
18.00 percent complete
19.00 percent complete
20.00 percent complete
21.00 percent complete
22.00 percent complete
23.00 percent complete
24.00 percent complete
25.00 percent complete
26.00 percent complete
27.00 percent complete
28.00 percent complete
29.00 percent complete
30.00 percent complete
31.00 percent complete
32.00 percent complete
33.00 percent complete
34.00 percent complete
35.00 percent complete
36.00 percent complete
37.00 percent complete
38.00 percent complete
39.00 percent complete
40.00 percent complete
41.00 percent complete
42.00 percent complete
43.00 percent complet